In [0]:
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("src_container", "")
dbutils.widgets.text("dst_container", "")
dbutils.widgets.text("sa_name", "")

In [0]:
from dataclasses import dataclass

@dataclass 
class TableConfig:
    table_name: str
    src_container: str
    dst_container: str
    sa_name: str

    @property
    def src_path(self):
        return f"abfss://{self.src_container}@{self.sa_name}.dfs.core.windows.net/{self.table_name}"
    
    @property 
    def dst_path(self):
        return f"abfss://{self.dst_container}@{self.sa_name}.dfs.core.windows.net/{self.table_name}"
    
    @property
    def schema_location(self):
        return f"/Volumes/dbw_main_catalog/bronze/vm_bronze/checkpoints/{self.table_name}/schemaLocation/"
    
    @property 
    def checkpoint_location(self):
        return f"/Volumes/dbw_main_catalog/bronze/vm_bronze/checkpoints/{self.table_name}/checkpointLocation/"

config = TableConfig(
    table_name=dbutils.widgets.get("table_name"),
    src_container=dbutils.widgets.get("src_container"),
    dst_container=dbutils.widgets.get("dst_container"),
    sa_name=dbutils.widgets.get("sa_name")
)

for k, v in config.__dict__.items():
    if not v: raise ValueError(f"Missing required parameter: {k}")

In [0]:
df = (
    spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", config.schema_location)
        .load(config.src_path)
)

In [0]:
query = (   
    df.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", config.checkpoint_location)
        .option("path", config.dst_path)
        .trigger(availableNow=True)
        .start()
)

query.awaitTermination()